In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import requests

In [ ]:
requests.get("http://localhost:11434").content

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

Model='llama3.1'

In [ ]:
links = fetch_website_links("https://nytimes.com")
links

In [ ]:
#Use lamma3.1 to read the link son the webpage and exrtract usefull links in json format

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

#user_prompt += "text"
#user_prompt = user_prompt + "text"

In [ ]:
print(get_links_user_prompt("https://nytimes.com"))

In [9]:
def select_relevant_links(url):
    response =  ollama.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)          # strings to dict
    return links
    

In [10]:
select_relevant_links("https://nytimes.com")

{'url': [24,
  {'name': 'puzzles', 'domain': 'nytimes.com', 'path': '/games/connections'},
  'name',
  'https://wordsNYTimes.com/games/spellingbee/game',
  'help.nytimes.com/hc/en-us/articles/"15014792127-Copyright-Notice',
  'https://www.nytco.com/>/careers',
  'help.nytimes.com/hc/en-us/articles/115015727108-Accessiblity',
  'http:>puzzlesNYTimes.com/game/daily-crossword',
  'help.nytimes.com/hc/en-us/articles/10940941449492-The-New-York-Times-&Company Privacy Policy',
  '/path/to/sitemap/',
  'https://www.nytco.com/>/carrers']}

In [ ]:
select_relevant_links("https://cricbuzz.com")

: 

: 

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {Model}")
    response = ollama.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(links)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://cricbuzz.com")

In [ ]:
select_relevant_links("https://huggingface.co")

In [13]:
#Assemble all the details into another prompt to ollama
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customersand careers/jobs if you have the information.
"""

# # Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.1
{'links': [{'type': 'company page', 'url': '/'}, {'type': 'what we offer models', 'url': '/models'}, {'type': 'datasets and resources', 'url': '/datasets'}, {'type': 'AI platform for building spaces', 'url': '/spaces'}, {'type': 'docs/transformers', 'url': 'https://huggingface.co/docs/transformers'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'who we are', 'url': 'https://github.com/huggingface'}, {'type': 'social media links', 'url': 'https://twitter.com/huggingface'}, {'type': 'careers/jobs', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}]}
Found 10 relevant links


MissingSchema: Invalid URL '/': No scheme supplied. Perhaps you meant https:///?

In [20]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.1
{'links': [{'type': 'home', 'url': 'https://huggingface.co/'}, {'type': 'models', 'url': 'https://huggingface.co/models'}, {'type': 'datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces', 'url': 'https://huggingface.co/spaces'}, {'type': 'docs', 'url': 'https://huggingface.co/docs/'}, {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing#endpoints'}, {'type': 'inference/models', 'url': 'https://huggingface.co/inference/models'}, {'type': 'learn', 'url': 'https://huggingface.co/learn'}, {'type': 'blog', 'url': 'https://discuss.huggingface.co'}, {'type': 'status', 'url': 'https://status.huggingface.co/'}]}
Found 11 relevant links


**Hugging Face: Accelerating the Machine Learning Ecosystem**

Hugging Face is an open-source platform that empowers the machine learning community to collaborate, build, and share models, datasets, and applications. By harnessing the power of collective innovation, Hugging Face speeds up research, development, and commercialization in AI.

**What We Do**

* **Host and Collaborate on Unlimited Public Models, Datasets, and Applications**: Our platform facilitates seamless collaboration between researchers, developers, and organizations.
* **Accelerate Machine Learning Development**: With our open-source stack, you can move faster in your ML journey, leveraging our vast collection of pretrained models, datasets, and application templates.
* **Explore All Modality**: From Text to Image, Video, Audio, or 3D, our platform supports the development and deployment of AI solutions across various industries.

**Our Vision**

At Hugging Face, we envision a future where AI is harnessed for the betterment of humanity. By fostering innovation, collaboration, and open dialogue, we create an inclusive environment for everyone to participate in shaping the future of machine learning.

## **Become Part of Our Community**

*   Collaborate on models, datasets, and applications
*   Join discussions and forums for expert knowledge-sharing
*   Participate in hackathons and events

## **Explore Careers at Hugging Face**

We're a company that's passionate about creating an inclusive space for everyone to thrive. We welcome individuals from diverse backgrounds and experiences who share our vision of building a better future through AI.

### Join Our Team

As a global community, we are always looking for talented professionals in various fields:

*   Software Development (ML, AI, Web)
*   Research and Data Science
*   Business and Marketing
*   Design and UX

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True  # streanms one by one in chunks (parts)
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
import gradio as gr

In [ ]:
def generate_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True  # streanms one by one in chunks (parts)
    )    
    response = ""
    
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield  response

In [25]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=generate_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co"],
            ["Cricbuzz", "https://cricbuzz.com"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
astream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
#Humorous system prompt
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
stream_brochure("CricBuzz", "https://nationalgeographic.com")

In [ ]:
stream_brochure("VU pune", "https://vupune.ac.in")